# 미끄러운 미로: 사전학습 → SFT → GRPO

각 단계를 학습한 뒤 직접 실행해 보고 다음 단계로 넘어갑니다.
**‘모두 실행’ 대신 필요한 셀을 순서대로 실행하세요.**

- 사전학습 후: 지정한 행동의 다음 위치 확률을 예측합니다.
- SFT 후: BFS 행동을 배운 모델이 목표로 이동합니다.
- GRPO 후: SFT와 같은 지도에서 실제 위험을 고려한 행동을 비교합니다.

[규칙·단계별 목표](https://github.com/HisameOgasahara/tmp_full_Training/blob/main/docs/학습설계.md)
· [실제 검증 범위](https://github.com/HisameOgasahara/tmp_full_Training/blob/main/docs/검증결과.md)

T4용으로 설계한 작은 실험입니다. 전체 학습의 수렴이나 GRPO 개선은 보장되지 않습니다.

## 0. 최신 코드 준비

Colab 런타임을 GPU로 설정한 뒤 실행하세요. 이 셀은 실행할 때마다 최신 `main`을
새 임시 폴더에 복제합니다. **준비 완료 후 표시되는 커밋**이 실제 실행 코드입니다.
학습 중에는 이 셀을 다시 실행하지 마세요. Colab에 설치된 PyTorch는 유지합니다.

In [ ]:
import importlib
import io
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.request

REPO_URL = "https://github.com/HisameOgasahara/tmp_full_Training.git"
REPO_ROOT = Path(tempfile.mkdtemp(prefix="slippery_maze_")) / "repository"
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_ROOT)], check=True)
REVISION = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()

uv_executable = shutil.which("uv")
if uv_executable is None:
    if not sys.platform.startswith("linux"):
        raise RuntimeError("로컬에서는 uv를 설치한 뒤 이 셀을 실행하세요. Colab에서는 자동 준비됩니다.")
    archive_url = "https://github.com/astral-sh/uv/releases/download/0.12.23/uv-x86_64-unknown-linux-gnu.tar.gz"
    data = urllib.request.urlopen(archive_url, timeout=120).read()
    with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as archive:
        member = next(item for item in archive.getmembers() if item.name.endswith("/uv") and item.isfile())
        uv_path = REPO_ROOT.parent / "uv"
        uv_path.write_bytes(archive.extractfile(member).read())
    uv_path.chmod(0o755)
    uv_executable = str(uv_path)

import torch
subprocess.run([uv_executable, "pip", "install", "--system", "--python", sys.executable,
                "numpy>=1.26,<3", "matplotlib>=3.8,<4", "ipywidgets>=8.1,<9"], check=True)
subprocess.run([uv_executable, "pip", "install", "--system", "--python", sys.executable,
                "--no-deps", "--editable", str(REPO_ROOT)], check=True)
for name in list(sys.modules):
    if name == "maze_training" or name.startswith("maze_training."):
        del sys.modules[name]
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
importlib.invalidate_caches()
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
print("실제 실행 커밋:", REVISION)
print("코드 위치:", REPO_ROOT)
print("PyTorch:", torch.__version__)
print("장치:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (t4 프로필은 GPU 권장)")

### 설정

`t4`는 학습용, `smoke`는 수초~수분의 연결 확인용입니다. smoke 결과로 성능을 판단하지 마세요.
`RESUME = True`는 **이미 체크포인트가 있는 같은 단계**를 이어 학습할 때 사용합니다.
새 단계로 넘어갈 때는 `False`로 돌려놓으세요. 같은 단계의 새 학습은 기존 파일을 덮어씁니다.

In [ ]:
PROFILE = "t4"  # "smoke"로 바꾸면 연결 확인용 짧은 실행
RESUME = False
CONFIG_PATH = str(REPO_ROOT / "configs" / f"{PROFILE}.json")
OUTPUT_ROOT = Path("/content/maze_runs") if Path("/content").exists() else REPO_ROOT / "runs"
OUTPUT_DIR = OUTPUT_ROOT / PROFILE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from maze_training.training import run_stage
from maze_training.notebook import report_world, report_navigation, show_training_curve, launch_world, launch_navigation
from maze_training.runtime import read_config, seed_runtime, create_model, choose_device, save_checkpoint

config = read_config(CONFIG_PATH)
model_for_size, _ = create_model(config, choose_device())
print("파라미터:", f"{sum(p.numel() for p in model_for_size.parameters()):,}")
print("설정:", CONFIG_PATH)
print("체크포인트:", OUTPUT_DIR)
del model_for_size

### 선택: Google Drive에 저장

런타임이 사라져도 모델을 유지하려면 **학습 전에** 아래 셀의 `USE_DRIVE`를 `True`로 바꾸세요.
Drive 연결은 사용자가 Colab에서 승인합니다. 기본 `/content` 저장 파일은 런타임 종료 시 사라질 수 있습니다.

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/maze_training") / PROFILE
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("저장 위치:", OUTPUT_DIR)

## 1. 사전학습 — 행동의 결과를 예측

목표: 지도·현재 위치·행동으로 다음 위치의 확률 분포를 예측합니다.
벽 충돌과 미끄러짐의 규칙을 배웁니다. 이 단계에는 BFS 행동 정답이 없습니다.

In [ ]:
PRETRAIN_PATH = run_stage("pretrain", CONFIG_PATH, OUTPUT_DIR, resume=RESUME)
show_training_curve(OUTPUT_DIR, "pretrain")

### 1단계 평가와 직접 실행

TV distance와 KL은 낮을수록 좋습니다. 미끄러운 칸의 결과는 확률적이므로
단일 샘플을 맞혔는지만으로 판단하지 마세요. 화면에서 행동을 선택하고 예측·실제 이동을 비교합니다.
**현재 모델은 행동 선택을 배우지 않았습니다.** 다음 단계는 확인 후 직접 실행하세요.

In [ ]:
PRETRAIN_PATH = str(OUTPUT_DIR / "pretrain.pt")
world_report = report_world(PRETRAIN_PATH)
world_demo = launch_world(PRETRAIN_PATH)

## 2. SFT — 목표로 이동하는 행동을 모방

목표: BFS가 선택한 다음 행동을 배웁니다. BFS는 벽·구멍을 피하지만 미끄러짐은 무시합니다.
같은 설정의 사전학습 체크포인트를 불러옵니다. 지도에서 여러 현재 위치를 학습합니다.

In [ ]:
SFT_PATH = run_stage("sft", CONFIG_PATH, OUTPUT_DIR, resume=RESUME)
show_training_curve(OUTPUT_DIR, "sft")

### 2단계 평가와 직접 실행

먼저 미끄러짐을 끈 조건에서, 이어 실제 미끄러짐 조건에서 평가합니다.
실행 화면의 ‘미끄러짐 켜기’를 바꾼 뒤 ‘지도 초기화’를 누르세요.
그림 한 번보다 여러 지도의 도착률·추락률·평균 보상을 함께 보세요.

In [ ]:
SFT_PATH = str(OUTPUT_DIR / "sft.pt")
sft_dry_report = report_navigation({"SFT": SFT_PATH}, CONFIG_PATH, slippery=False)
sft_report = report_navigation({"SFT": SFT_PATH}, CONFIG_PATH, slippery=True)
sft_demo = launch_navigation({"SFT": SFT_PATH}, CONFIG_PATH)

## 3. GRPO — 실제 성공과 위험을 고려

목표: 실제 환경에서 후보 에피소드를 실행하고 도착·추락·이동·충돌 보상으로 행동을 개선합니다.
SFT 모델에서 시작하며 BFS나 최적 정책의 정답 행동을 사용하지 않습니다.
`useful_group_fraction`이 계속 0이면 후보 보상이 모두 같아 학습 신호가 부족한 상태입니다.

In [ ]:
GRPO_PATH = run_stage("grpo", CONFIG_PATH, OUTPUT_DIR, resume=RESUME)
show_training_curve(OUTPUT_DIR, "grpo")

### 3단계 평가와 나란히 실행

같은 지도와 환경 난수에서 SFT·GRPO를 비교합니다. 화면은 argmax 행동으로 실행합니다.
GRPO 학습의 샘플링 보상과 이 평가 결과는 다를 수 있습니다.
위험한 지름길 예시는 설명용이며, 모델이 우회하는 것을 보장하지 않습니다.

In [ ]:
SFT_PATH, GRPO_PATH = str(OUTPUT_DIR / "sft.pt"), str(OUTPUT_DIR / "grpo.pt")
comparison = report_navigation({"SFT": SFT_PATH, "GRPO": GRPO_PATH}, CONFIG_PATH)
comparison_demo = launch_navigation({"SFT": SFT_PATH, "GRPO": GRPO_PATH}, CONFIG_PATH)

## 4. 최종 테스트

학습·설정 선택을 마친 뒤 실행하세요. 검증과 다른 지도 seed를 사용합니다.
BFS는 매 이동 후 다시 계획하고, optimal은 실제 미끄러짐·보상·시간 제한을 고려한 정책입니다.
GRPO 결과가 나쁘면 그 사실을 그대로 기록하세요. 세 단계 효과를 확인한 결과는 사전에 정해져 있지 않습니다.

In [ ]:
final_test = report_navigation({"SFT": str(OUTPUT_DIR / "sft.pt"), "GRPO": str(OUTPUT_DIR / "grpo.pt")}, CONFIG_PATH, split="test")

## 5. 선택적 단계 생략 비교

추가 학습 비용이 발생하므로 각각의 스위치를 켰을 때만 실행합니다.
사전학습 생략은 무작위 초기화 → SFT → GRPO,
SFT 생략은 사전학습 → GRPO입니다. 출력 폴더를 분리해 전체 흐름의 모델을 보존합니다.
같은 후속 단계 예산에서 검증 지표와 학습 시간을 비교하세요. 총 계산량이 동일한 비교는 아닙니다.

In [ ]:
RUN_WITHOUT_PRETRAIN = False
if RUN_WITHOUT_PRETRAIN:
    ablation_dir = OUTPUT_DIR / "without_pretrain"
    seed_runtime(config["seed"])
    random_model, _ = create_model(config, choose_device())
    random_path = ablation_dir / "random_initialization.pt"
    save_checkpoint(random_path, random_model, config, "random_initialization", 0)
    del random_model
    run_stage("sft", CONFIG_PATH, ablation_dir, initialize_from=random_path)
    run_stage("grpo", CONFIG_PATH, ablation_dir)
    report_navigation({"전체 흐름": str(OUTPUT_DIR / "grpo.pt"), "사전학습 생략": str(ablation_dir / "grpo.pt")}, CONFIG_PATH)

In [ ]:
RUN_WITHOUT_SFT = False
if RUN_WITHOUT_SFT:
    ablation_dir = OUTPUT_DIR / "without_sft"
    run_stage("grpo", CONFIG_PATH, ablation_dir, initialize_from=OUTPUT_DIR / "pretrain.pt")
    report_navigation({"전체 흐름": str(OUTPUT_DIR / "grpo.pt"), "SFT 생략": str(ablation_dir / "grpo.pt")}, CONFIG_PATH)

## 6. 체크포인트 다운로드

런타임 종료 전에 필요한 모델을 저장하세요. ZIP에는 모델·설정·학습 로그·평가 결과만 들어갑니다.
다음 런타임에서 동일한 출력 폴더에 복원하면 실행 화면을 다시 사용할 수 있습니다.

In [ ]:
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from google.colab import files
    archive_path = shutil.make_archive(str(OUTPUT_DIR.parent / f"maze_results_{PROFILE}"), "zip", OUTPUT_DIR)
    files.download(archive_path)